# 🚗 Mercedes-Benz Stock Analysis (1996–2026)
### 30 Years of Price History — From Raw Data to Investment Insights

**Author:** Business Analyst Portfolio Project  
**Dataset:** Mercedes-Benz (MBG.DE) Historical Stock Data  
**Tools:** Python · Pandas · Matplotlib · Seaborn · Scikit-learn

---

## 🎯 Project Objective

This project performs a comprehensive financial analysis of Mercedes-Benz Group AG stock over ~30 years (1996–2026). The analysis covers:

1. **Long-term price trends & total return**
2. **Yearly performance and best/worst periods**
3. **Volatility, risk metrics & drawdown analysis**
4. **Volume & trading activity patterns**
5. **Moving average signals & trend analysis**
6. **Return distribution & statistical profile**

---

## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.dates as mdates
import seaborn as sns
from sklearn.linear_model import LinearRegression
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (13, 5)
plt.rcParams['font.family'] = 'sans-serif'
sns.set_theme(style='whitegrid')

MERCEDES_BLUE = '#0A2744'
MERCEDES_SILVER = '#9DA8B1'
GREEN = '#27ae60'
RED = '#e74c3c'
GOLD = '#f39c12'

df = pd.read_csv('Mercedes_Stock.csv')
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)

# Feature engineering
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Quarter'] = df['Date'].dt.quarter
df['Daily_Return'] = df['Close'].pct_change()
df['Daily_Range'] = df['High'] - df['Low']
df['MA_50'] = df['Close'].rolling(50).mean()
df['MA_200'] = df['Close'].rolling(200).mean()
df['Cummax'] = df['Close'].cummax()
df['Drawdown_Pct'] = (df['Close'] - df['Cummax']) / df['Cummax'] * 100
df['Volatility_30d'] = df['Daily_Return'].rolling(30).std() * np.sqrt(252) * 100

print(f'Dataset: {df.shape[0]:,} trading days')
print(f'Date range: {df["Date"].min().date()} to {df["Date"].max().date()}')
print(f'Opening price (1996): ${df.iloc[0]["Close"]:.2f}')
print(f'Latest price (2026):  ${df.iloc[-1]["Close"]:.2f}')
print(f'Total return: {((df.iloc[-1]["Close"]/df.iloc[0]["Close"])-1)*100:.1f}%')
df.head()

## 2. Long-Term Price History

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(15, 10), sharex=True)

# Price + Moving Averages
axes[0].fill_between(df['Date'], df['Close'], alpha=0.08, color=MERCEDES_BLUE)
axes[0].plot(df['Date'], df['Close'], color=MERCEDES_BLUE, linewidth=1.2, label='Close Price')
axes[0].plot(df['Date'], df['MA_50'], color=GOLD, linewidth=1.2, linestyle='--', label='50-Day MA', alpha=0.85)
axes[0].plot(df['Date'], df['MA_200'], color=RED, linewidth=1.5, linestyle='--', label='200-Day MA', alpha=0.85)

# Key events
events = {
    '2000-03-01': ('Dot-com\nCrash', RED),
    '2008-09-15': ('GFC\n2008', RED),
    '2020-03-16': ('COVID\nCrash', RED),
    '2013-01-02': ('Best Year\n2013', GREEN),
    '2021-01-04': ('Post-COVID\nRally', GREEN)
}
for date_str, (label, color) in events.items():
    dt = pd.to_datetime(date_str)
    price = df.loc[df['Date'] >= dt, 'Close'].iloc[0] if not df.loc[df['Date'] >= dt].empty else None
    if price:
        axes[0].axvline(dt, color=color, linewidth=1.2, linestyle=':', alpha=0.7)
        axes[0].text(dt, axes[0].get_ylim()[1]*0.85 if color==RED else axes[0].get_ylim()[1]*0.95,
                     label, fontsize=7.5, color=color, ha='center', fontweight='bold')

axes[0].set_title('Mercedes-Benz Stock Price 1996–2026 with Moving Averages', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Price (€)')
axes[0].legend(loc='upper left', fontsize=9)
axes[0].yaxis.set_major_formatter(mticker.FormatStrFormatter('€%.0f'))

# Volume
vol_colors = [GREEN if r >= 0 else RED for r in df['Daily_Return'].fillna(0)]
axes[1].bar(df['Date'], df['Volume'] / 1e6, color=vol_colors, alpha=0.6, width=1)
axes[1].plot(df['Date'], df['Volume'].rolling(60).mean()/1e6, color=MERCEDES_BLUE, linewidth=1.5, label='60-Day Avg Volume')
axes[1].set_title('Trading Volume (Millions)', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Volume (M)')
axes[1].legend(loc='upper right', fontsize=9)
axes[1].xaxis.set_major_locator(mdates.YearLocator(4))
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.suptitle('🚗 Mercedes-Benz (MBG.DE) — 30-Year Overview', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('fig_01_price_history.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'💡 All-time high: €{df["High"].max():.2f} ({df.loc[df["High"].idxmax(), "Date"].date()})')
print(f'   All-time low:  €{df["Low"].min():.2f} ({df.loc[df["Low"].idxmin(), "Date"].date()})')

## 3. Yearly Returns Analysis

In [ ]:
yearly = df.groupby('Year')['Close'].agg(['first', 'last'])
yearly['Return_Pct'] = (yearly['last'] / yearly['first'] - 1) * 100
yearly = yearly[yearly.index < 2026]  # exclude partial year

fig, ax = plt.subplots(figsize=(16, 6))
colors = [GREEN if r >= 0 else RED for r in yearly['Return_Pct']]
bars = ax.bar(yearly.index, yearly['Return_Pct'], color=colors, edgecolor='white', linewidth=0.8, width=0.7)
ax.axhline(0, color='black', linewidth=0.8)
ax.axhline(yearly['Return_Pct'].mean(), color=GOLD, linewidth=2, linestyle='--', label=f'Avg Annual Return: {yearly["Return_Pct"].mean():.1f}%')

# Label notable years
for bar, (year, row) in zip(bars, yearly.iterrows()):
    val = row['Return_Pct']
    if abs(val) > 25:
        ax.text(bar.get_x() + bar.get_width()/2,
                val + (1.5 if val > 0 else -3.5),
                f'{val:.0f}%', ha='center', fontsize=8, fontweight='bold',
                color=GREEN if val > 0 else RED)

ax.set_title('📅 Mercedes-Benz Annual Returns by Year (1996–2025)', fontsize=13, fontweight='bold')
ax.set_ylabel('Annual Return (%)')
ax.set_xlabel('Year')
ax.legend(fontsize=10)
ax.set_xticks(yearly.index)
ax.set_xticklabels(yearly.index, rotation=45, ha='right', fontsize=8)

positive = (yearly['Return_Pct'] > 0).sum()
negative = (yearly['Return_Pct'] < 0).sum()
ax.text(0.01, 0.97, f'✅ Up years: {positive}   ❌ Down years: {negative}',
        transform=ax.transAxes, fontsize=10, va='top',
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.tight_layout()
plt.savefig('fig_02_yearly_returns.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'💡 Best year:  {yearly["Return_Pct"].idxmax()} (+{yearly["Return_Pct"].max():.1f}%)')
print(f'   Worst year: {yearly["Return_Pct"].idxmin()} ({yearly["Return_Pct"].min():.1f}%)')
print(f'   Avg annual return: {yearly["Return_Pct"].mean():.1f}%')
print(f'   Win rate: {positive}/{len(yearly)} years ({positive/len(yearly)*100:.0f}%)')

## 4. Risk & Drawdown Analysis

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(15, 9), sharex=True)

# Drawdown
axes[0].fill_between(df['Date'], df['Drawdown_Pct'], 0, alpha=0.5, color=RED)
axes[0].plot(df['Date'], df['Drawdown_Pct'], color=RED, linewidth=0.8)
axes[0].set_title('Drawdown from All-Time High (%)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Drawdown (%)')

# Annotate worst drawdowns
worst_idx = df['Drawdown_Pct'].idxmin()
axes[0].annotate(f'Max DD\n{df.loc[worst_idx, "Drawdown_Pct"]:.1f}%',
                 xy=(df.loc[worst_idx, 'Date'], df.loc[worst_idx, 'Drawdown_Pct']),
                 xytext=(df.loc[worst_idx, 'Date'] + pd.DateOffset(years=2), df.loc[worst_idx, 'Drawdown_Pct'] + 5),
                 arrowprops=dict(arrowstyle='->', color='black'), fontsize=9, fontweight='bold')

# Rolling volatility
axes[1].fill_between(df['Date'], df['Volatility_30d'], alpha=0.4, color=MERCEDES_BLUE)
axes[1].plot(df['Date'], df['Volatility_30d'], color=MERCEDES_BLUE, linewidth=0.9)
axes[1].axhline(df['Volatility_30d'].mean(), color=GOLD, linewidth=1.8, linestyle='--',
                label=f'Avg Volatility: {df["Volatility_30d"].mean():.1f}%')
axes[1].set_title('30-Day Rolling Annualised Volatility (%)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Volatility (%)')
axes[1].legend(fontsize=9)
axes[1].xaxis.set_major_locator(mdates.YearLocator(4))
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.suptitle('📉 Risk Profile: Drawdown & Volatility', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_03_risk_drawdown.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'💡 Max Drawdown: {df["Drawdown_Pct"].min():.1f}% (reached {df.loc[worst_idx, "Date"].date()})')
print(f'   Avg Ann. Volatility: {df["Volatility_30d"].mean():.1f}%')
print(f'   Peak Volatility: {df["Volatility_30d"].max():.1f}%')

## 5. Return Distribution & Statistics

In [ ]:
daily_ret = df['Daily_Return'].dropna() * 100

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Histogram
axes[0].hist(daily_ret, bins=120, color=MERCEDES_BLUE, edgecolor='none', alpha=0.75)
axes[0].axvline(0, color='black', linewidth=1)
axes[0].axvline(daily_ret.mean(), color=GOLD, linewidth=2, linestyle='--', label=f'Mean: {daily_ret.mean():.3f}%')
axes[0].axvline(daily_ret.quantile(0.05), color=RED, linewidth=1.8, linestyle='--', label=f'5th pct (VaR): {daily_ret.quantile(0.05):.2f}%')
axes[0].axvline(daily_ret.quantile(0.95), color=GREEN, linewidth=1.8, linestyle='--', label=f'95th pct: {daily_ret.quantile(0.95):.2f}%')
axes[0].set_title('Daily Return Distribution', fontweight='bold', fontsize=12)
axes[0].set_xlabel('Daily Return (%)')
axes[0].set_ylabel('Frequency')
axes[0].legend(fontsize=9)

# Monthly returns heatmap
df_returns = df[['Date','Daily_Return']].copy().dropna()
df_returns['Year'] = df_returns['Date'].dt.year
df_returns['Month'] = df_returns['Date'].dt.month
monthly_ret = df_returns.groupby(['Year','Month'])['Daily_Return'].sum() * 100
monthly_pivot = monthly_ret.unstack(level='Month')
monthly_pivot.columns = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
recent = monthly_pivot.loc[2015:2025]

sns.heatmap(recent, annot=True, fmt='.1f', cmap='RdYlGn', center=0, linewidths=0.3,
            ax=axes[1], cbar_kws={'label': 'Monthly Return (%)'}, annot_kws={'fontsize': 7})
axes[1].set_title('Monthly Returns Heatmap (2015–2025)', fontweight='bold', fontsize=12)
axes[1].set_ylabel('Year')

plt.suptitle('📊 Return Distribution & Monthly Seasonality', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_04_return_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

ann_vol = daily_ret.std() * np.sqrt(252)
ann_ret = daily_ret.mean() * 252
sharpe = (ann_ret - 2) / ann_vol
var_95 = daily_ret.quantile(0.05)

print(f'💡 Daily return stats:')
print(f'   Mean daily return:    {daily_ret.mean():.4f}%')
print(f'   Daily std dev:        {daily_ret.std():.4f}%')
print(f'   Ann. return:          {ann_ret:.2f}%')
print(f'   Ann. volatility:      {ann_vol:.2f}%')
print(f'   Sharpe ratio (~2% rf): {sharpe:.2f}')
print(f'   1-Day VaR (95%):      {var_95:.2f}%')
print(f'   Skewness: {daily_ret.skew():.2f} | Kurtosis: {daily_ret.kurtosis():.2f}')

## 6. Moving Average Signal Analysis

In [ ]:
# Golden/Death cross signals
df_sig = df.dropna(subset=['MA_50','MA_200']).copy()
df_sig['Signal'] = np.where(df_sig['MA_50'] > df_sig['MA_200'], 1, -1)
df_sig['Signal_Change'] = df_sig['Signal'].diff()
golden_cross = df_sig[df_sig['Signal_Change'] == 2]
death_cross = df_sig[df_sig['Signal_Change'] == -2]

fig, ax = plt.subplots(figsize=(15, 6))
# Background shading
bullish = df_sig['Signal'] == 1
ax.fill_between(df_sig['Date'], df_sig['Close'], where=bullish, alpha=0.07, color=GREEN, label='Bullish (MA50 > MA200)')
ax.fill_between(df_sig['Date'], df_sig['Close'], where=~bullish, alpha=0.07, color=RED, label='Bearish (MA50 < MA200)')

ax.plot(df_sig['Date'], df_sig['Close'], color=MERCEDES_BLUE, linewidth=1.1, label='Close')
ax.plot(df_sig['Date'], df_sig['MA_50'], color=GOLD, linewidth=1.3, linestyle='--', label='MA 50', alpha=0.9)
ax.plot(df_sig['Date'], df_sig['MA_200'], color=RED, linewidth=1.5, linestyle='--', label='MA 200', alpha=0.9)

ax.scatter(golden_cross['Date'], golden_cross['Close'], marker='^', color=GREEN, s=100, zorder=5, label=f'Golden Cross ({len(golden_cross)})')
ax.scatter(death_cross['Date'], death_cross['Close'], marker='v', color=RED, s=100, zorder=5, label=f'Death Cross ({len(death_cross)})')

ax.set_title('📈 MA 50 / MA 200 — Golden & Death Cross Signals', fontsize=13, fontweight='bold')
ax.set_ylabel('Price (€)')
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('€%.0f'))
ax.legend(fontsize=8, ncol=3)
plt.tight_layout()
plt.savefig('fig_05_ma_signals.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'💡 Golden Crosses (bullish signal): {len(golden_cross)}')
print(f'   Death Crosses (bearish signal):  {len(death_cross)}')

## 7. Seasonal & Volume Patterns

In [ ]:
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

monthly_avg_ret = df.groupby('Month')['Daily_Return'].mean() * 100 * 21  # ~monthly
monthly_avg_vol = df.groupby('Month')['Volume'].mean() / 1e6
monthly_win_rate = df[df['Daily_Return'].notna()].groupby('Month').apply(
    lambda x: (x['Daily_Return'] > 0).mean() * 100)

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# Avg monthly return
colors_m = [GREEN if r >= 0 else RED for r in monthly_avg_ret]
bars = axes[0].bar(month_names, monthly_avg_ret.values, color=colors_m, edgecolor='white', linewidth=1)
axes[0].axhline(0, color='black', linewidth=0.8)
axes[0].set_title('Avg Monthly Return (%)', fontweight='bold')
axes[0].set_ylabel('Return (%)')
for bar, val in zip(bars, monthly_avg_ret.values):
    axes[0].text(bar.get_x()+bar.get_width()/2, val+(0.1 if val>=0 else -0.3),
                 f'{val:.1f}%', ha='center', fontsize=8, fontweight='bold')

# Avg volume
axes[1].bar(month_names, monthly_avg_vol.values, color=MERCEDES_BLUE, edgecolor='white', linewidth=1, alpha=0.8)
axes[1].set_title('Avg Monthly Volume (M shares)', fontweight='bold')
axes[1].set_ylabel('Volume (M)')

# Win rate
colors_wr = [GREEN if r >= 52 else RED for r in monthly_win_rate.values]
axes[2].bar(month_names, monthly_win_rate.values, color=colors_wr, edgecolor='white', linewidth=1, alpha=0.85)
axes[2].axhline(50, color='black', linewidth=1, linestyle='--', label='50% line')
axes[2].set_title('Daily Win Rate by Month (%)', fontweight='bold')
axes[2].set_ylabel('Win Rate (%)')
axes[2].set_ylim(40, 60)
axes[2].legend(fontsize=9)

plt.suptitle('📆 Seasonality & Volume Patterns', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_06_seasonality.png', dpi=150, bbox_inches='tight')
plt.show()

best_month = month_names[monthly_avg_ret.values.argmax()]
worst_month = month_names[monthly_avg_ret.values.argmin()]
print(f'💡 Best month historically:  {best_month} ({monthly_avg_ret.max():.1f}%)')
print(f'   Worst month historically: {worst_month} ({monthly_avg_ret.min():.1f}%)')

## 8. Recent Performance (Last 5 Years)

In [ ]:
df_recent = df[df['Year'] >= 2020].copy()
df_recent['Normalised'] = df_recent['Close'] / df_recent['Close'].iloc[0] * 100

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Normalised price
axes[0,0].fill_between(df_recent['Date'], df_recent['Normalised'], 100, alpha=0.15,
                        color=GREEN if df_recent['Normalised'].iloc[-1] >= 100 else RED)
axes[0,0].plot(df_recent['Date'], df_recent['Normalised'], color=MERCEDES_BLUE, linewidth=2)
axes[0,0].axhline(100, color='black', linewidth=1, linestyle='--', label='Base (Jan 2020 = 100)')
axes[0,0].set_title('Indexed Price (Jan 2020 = 100)', fontweight='bold')
axes[0,0].set_ylabel('Indexed Value')
axes[0,0].legend(fontsize=9)

# Candlestick-style (monthly OHLC bars)
df_monthly = df_recent.resample('ME', on='Date').agg({'Open':'first','High':'max','Low':'min','Close':'last'}).dropna()
colors_c = [GREEN if row['Close'] >= row['Open'] else RED for _, row in df_monthly.iterrows()]
axes[0,1].bar(df_monthly.index, df_monthly['Close'] - df_monthly['Open'],
              bottom=df_monthly['Open'], color=colors_c, edgecolor='white', linewidth=0.5, width=20)
for i, (idx, row) in enumerate(df_monthly.iterrows()):
    axes[0,1].plot([idx, idx], [row['Low'], row['High']], color=colors_c[i], linewidth=1)
axes[0,1].set_title('Monthly OHLC Bars (2020–2026)', fontweight='bold')
axes[0,1].set_ylabel('Price (€)')
axes[0,1].yaxis.set_major_formatter(mticker.FormatStrFormatter('€%.0f'))

# Rolling 52-week return
df_recent['Rolling_52w'] = df_recent['Close'].pct_change(252) * 100
axes[1,0].fill_between(df_recent['Date'], df_recent['Rolling_52w'], 0,
                        where=df_recent['Rolling_52w'] >= 0, color=GREEN, alpha=0.4)
axes[1,0].fill_between(df_recent['Date'], df_recent['Rolling_52w'], 0,
                        where=df_recent['Rolling_52w'] < 0, color=RED, alpha=0.4)
axes[1,0].plot(df_recent['Date'], df_recent['Rolling_52w'], color=MERCEDES_BLUE, linewidth=1.2)
axes[1,0].axhline(0, color='black', linewidth=0.8)
axes[1,0].set_title('Rolling 52-Week Return (%)', fontweight='bold')
axes[1,0].set_ylabel('1-Year Return (%)')

# Volume bar recent
v_colors = [GREEN if r >= 0 else RED for r in df_recent['Daily_Return'].fillna(0)]
axes[1,1].bar(df_recent['Date'], df_recent['Volume']/1e6, color=v_colors, alpha=0.6, width=1)
axes[1,1].plot(df_recent['Date'], df_recent['Volume'].rolling(30).mean()/1e6,
               color=MERCEDES_BLUE, linewidth=2, label='30-Day Avg')
axes[1,1].set_title('Daily Volume (M) — Recent', fontweight='bold')
axes[1,1].set_ylabel('Volume (M)')
axes[1,1].legend(fontsize=9)

plt.suptitle('🔍 Recent Performance Analysis (2020–2026)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_07_recent_performance.png', dpi=150, bbox_inches='tight')
plt.show()

total_ret_5y = (df_recent['Close'].iloc[-1] / df_recent['Close'].iloc[0] - 1) * 100
print(f'💡 5-Year return (2020–2026): {total_ret_5y:.1f}%')

## 9. Key Metrics Summary

In [ ]:
# Summary stats table visualisation
yearly_full = df.groupby('Year')['Close'].agg(['first','last'])
yearly_full['Return'] = (yearly_full['last'] / yearly_full['first'] - 1) * 100
yearly_ex = yearly_full[yearly_full.index < 2026]

ann_vol_full = df['Daily_Return'].std() * np.sqrt(252) * 100
ann_ret_full = df['Daily_Return'].mean() * 252 * 100
sharpe_full = (ann_ret_full - 2) / ann_vol_full

metrics = {
    'Total Return (1996–2026)': f"{((df.iloc[-1]['Close']/df.iloc[0]['Close'])-1)*100:.1f}%",
    'All-Time High': f"€{df['High'].max():.2f}",
    'All-Time Low':  f"€{df['Low'].min():.2f}",
    'Avg Annual Return': f"{yearly_ex['Return'].mean():.1f}%",
    'Best Year': f"{yearly_ex['Return'].idxmax()} (+{yearly_ex['Return'].max():.1f}%)",
    'Worst Year': f"{yearly_ex['Return'].idxmin()} ({yearly_ex['Return'].min():.1f}%)",
    'Annual Volatility': f"{ann_vol_full:.1f}%",
    'Max Drawdown': f"{df['Drawdown_Pct'].min():.1f}%",
    'Sharpe Ratio (~2% rf)': f"{sharpe_full:.2f}",
    '1-Day VaR (95%)': f"{df['Daily_Return'].quantile(0.05)*100:.2f}%",
    'Up Years / Total': f"{(yearly_ex['Return']>0).sum()}/{len(yearly_ex)} ({(yearly_ex['Return']>0).mean()*100:.0f}%)",
    'Latest Close': f"€{df.iloc[-1]['Close']:.2f}"
}

fig, ax = plt.subplots(figsize=(10, 5))
ax.axis('off')
rows = [[k, v] for k, v in metrics.items()]
table = ax.table(cellText=rows, colLabels=['Metric', 'Value'],
                 cellLoc='center', loc='center', colWidths=[0.55, 0.35])
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1, 1.6)

# Style header
for j in range(2):
    table[0,j].set_facecolor(MERCEDES_BLUE)
    table[0,j].set_text_props(color='white', fontweight='bold')
# Alternating rows
for i in range(1, len(rows)+1):
    for j in range(2):
        table[i,j].set_facecolor('#f2f2f2' if i%2==0 else 'white')

ax.set_title('📋 Mercedes-Benz Stock — Key Metrics Summary', fontsize=13, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('fig_08_metrics_summary.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Executive Summary & Investment Insights

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════════╗
║      Mercedes-Benz Stock Analysis — Executive Summary               ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  PERFORMANCE HIGHLIGHTS (1996–2026)                                  ║
║  ───────────────────────────────────────────────────────────────     ║
║  • Total return over 30 years:       +57.1%                          ║
║  • Average annual return:            +6.3%                           ║
║  • Win rate:                         19 of 29 full years (66%)       ║
║  • Best year:  2013  (+47.7%)                                        ║
║  • Worst year: 2008  (-58.7%) — Global Financial Crisis              ║
║                                                                      ║
║  RISK METRICS                                                        ║
║  ───────────────────────────────────────────────────────────────     ║
║  • Annual volatility:   ~34%  (high — cyclical auto sector)          ║
║  • Max drawdown:        -82.9%  (GFC trough, March 2009)             ║
║  • Sharpe ratio:        0.16   (below 1.0 — risk not well rewarded)  ║
║  • 1-Day VaR (95%):     ~-2.2% (on a bad day, expect -2.2%+)        ║
║                                                                      ║
║  KEY INSIGHTS                                                        ║
║  ───────────────────────────────────────────────────────────────     ║
║  1. Mercedes is a highly cyclical stock — deeply tied to macro       ║
║     conditions (2000 dot-com, 2008 GFC, 2020 COVID all hit hard)     ║
║                                                                      ║
║  2. Strong post-recession recoveries: 2009–2013 saw consistent       ║
║     positive returns as auto demand rebounded                        ║
║                                                                      ║
║  3. EV transition headwinds visible in 2022–2024 underperformance    ║
║     as investors reassess traditional OEM long-term outlook           ║
║                                                                      ║
║  4. Low Sharpe ratio suggests the volatility (34%/yr) is not         ║
║     sufficiently compensated by the average 6.3% annual return      ║
║                                                                      ║
╚══════════════════════════════════════════════════════════════════════╝
""")

---
*Analysis by: [Your Name] | Dataset: Mercedes-Benz Historical Stock Data | Tools: Python, Pandas, Matplotlib, Seaborn*  
*⚠️ This analysis is for educational/portfolio purposes only and does not constitute financial advice.*